# gf_mul LDA Template Attack

논문 §3.2 Table 2 재현. 저장된 trace HDF5로부터 **6종 LDA template** 을 학습/평가한다.

- Target: Operand 0 / Operand 1 / Output
- Label: Value (256 클래스) / Hamming weight (9 클래스)
- 전체 sample point를 feature로 사용
- 같은 파일 안에서 train/val/test 분할 (기본 60/20/20)


## 1. Imports

In [ ]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score

## 2. Configuration

In [ ]:
# ===== user config =====
# 단일 H5 파일에서 train/val/test를 분할합니다
DATA_H5 = "/Users/ijaeyeon/chipwhisperer/jupyter/user/batches_gf_mul_CW308_STM32F4/10000trace_1400point_seed12.h5"

# Leave as None to auto-detect from the HDF5 file.
TRACE_KEY = None
A_KEY = None
B_KEY = None
V_KEY = None

# Train/Val/Test split ratios
TRAIN_RATIO = 0.6  # 60% training
VAL_RATIO = 0.2    # 20% validation
TEST_RATIO = 0.2   # 20% test
SPLIT_SEED = 42    # 재현 가능하게 하기 위한 seed

## 3. HDF5 key resolver

In [ ]:
def resolve_h5_keys(f, trace_key=None, a_key=None, b_key=None, v_key=None):
    keys = list(f.keys())

    def pick(user_key, candidates, name):
        if user_key is not None:
            if user_key not in keys:
                raise KeyError(
                    f"{name} key '{user_key}' not found. Available datasets: {keys}"
                )
            return user_key
        for cand in candidates:
            if cand in keys:
                return cand
        raise KeyError(
            f"Could not auto-detect {name} key. Available datasets: {keys}"
        )

    trace_key = pick(trace_key, ["traces", "trace", "waves", "wave"], "trace")
    a_key     = pick(a_key,     ["a", "op0", "operand0"], "a")
    b_key     = pick(b_key,     ["b", "op1", "operand1"], "b")
    v_key     = pick(v_key,     ["v", "out", "output"],   "v")
    return trace_key, a_key, b_key, v_key

## 4. Helpers

In [ ]:
def load_h5_dataset(h5_path, trace_key=None, a_key=None, b_key=None, v_key=None):
    with h5py.File(h5_path, "r") as f:
        keys = list(f.keys())
        print(f"[{h5_path}] datasets:", keys)
        trace_key, a_key, b_key, v_key = resolve_h5_keys(
            f, trace_key, a_key, b_key, v_key
        )
        print(f"[{h5_path}] resolved keys:",
              {"TRACE_KEY": trace_key, "A_KEY": a_key, "B_KEY": b_key, "V_KEY": v_key})

        traces = f[trace_key][:].astype(np.float64)
        a = f[a_key][:].astype(np.uint8)
        b = f[b_key][:].astype(np.uint8)
        v = f[v_key][:].astype(np.uint8)
    return traces, a, b, v

def hw8(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.uint8)
    return np.unpackbits(x[:, None], axis=1).sum(axis=1).astype(np.uint8)

def fit_zscore(X_train):
    mu = X_train.mean(axis=0, keepdims=True)
    sd = X_train.std(axis=0, keepdims=True)
    sd[sd == 0] = 1.0
    return mu, sd

def apply_zscore(X, mu, sd):
    return (X - mu) / sd

def split_train_val_test(traces, labels, train_ratio=0.6, val_ratio=0.2, seed=42):
    """같은 데이터를 train/val/test로 분할합니다."""
    n = len(traces)
    np.random.seed(seed)
    indices = np.random.permutation(n)

    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    train_idx = indices[:n_train]
    val_idx = indices[n_train:n_train + n_val]
    test_idx = indices[n_train + n_val:]

    X_train = traces[train_idx]
    X_val = traces[val_idx]
    X_test = traces[test_idx]

    y_train = labels[train_idx]
    y_val = labels[val_idx]
    y_test = labels[test_idx]

    return (X_train, y_train), (X_val, y_val), (X_test, y_test)

def train_val_test_on_same_data(
    traces: np.ndarray,
    labels: np.ndarray,
    train_ratio: float = 0.6,
    val_ratio: float = 0.2,
    seed: int = 42,
    solver: str = "svd",
):
    """같은 데이터셋을 train/val/test로 분할하고 학습 및 평가합니다."""
    (X_train, y_train), (X_val, y_val), (X_test, y_test) = split_train_val_test(
        traces, labels, train_ratio, val_ratio, seed
    )

    mu, sd = fit_zscore(X_train)
    X_train_z = apply_zscore(X_train, mu, sd)
    X_val_z = apply_zscore(X_val, mu, sd)
    X_test_z = apply_zscore(X_test, mu, sd)

    clf = LinearDiscriminantAnalysis(solver=solver)
    clf.fit(X_train_z, y_train)

    pred_val = clf.predict(X_val_z)
    pred_test = clf.predict(X_test_z)

    acc_val = accuracy_score(y_val, pred_val)
    acc_test = accuracy_score(y_test, pred_test)

    return {
        "clf": clf,
        "mu": mu,
        "sd": sd,
        "pred_val": pred_val,
        "pred_test": pred_test,
        "acc_val": float(acc_val),
        "acc_test": float(acc_test),
        "n_train": int(len(y_train)),
        "n_val": int(len(y_val)),
        "n_test": int(len(y_test)),
        "n_classes": int(len(np.unique(y_train))),
    }

## 5. Load dataset

In [ ]:
traces, train_a, train_b, train_v = load_h5_dataset(
    DATA_H5, trace_key=TRACE_KEY, a_key=A_KEY, b_key=B_KEY, v_key=V_KEY
)

print("Total traces shape:", traces.shape)
print(f"Will split into: train ({int(len(traces)*TRAIN_RATIO)}), "
      f"val ({int(len(traces)*VAL_RATIO)}), "
      f"test ({int(len(traces)*TEST_RATIO)})")

## 6. Prepare labels

In [ ]:
train_a_u8 = train_a.astype(np.uint8)
train_b_u8 = train_b.astype(np.uint8)
train_v_u8 = train_v.astype(np.uint8)

train_a_hw = hw8(train_a_u8)
train_b_hw = hw8(train_b_u8)
train_v_hw = hw8(train_v_u8)

print("Prepared labels:")
print(f"  a (value): {len(np.unique(train_a_u8))} unique values")
print(f"  b (value): {len(np.unique(train_b_u8))} unique values")
print(f"  v (value): {len(np.unique(train_v_u8))} unique values")
print(f"  a (HW): {len(np.unique(train_a_hw))} unique values")
print(f"  b (HW): {len(np.unique(train_b_hw))} unique values")
print(f"  v (HW): {len(np.unique(train_v_hw))} unique values")

## 7. Run 6 LDA templates

In [ ]:
results = []

# -------------------------
# Value templates
# -------------------------
print("\n[Value template] Operand 0 (a)...")
res_val_a = train_val_test_on_same_data(
    traces, train_a_u8,
    train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO, seed=SPLIT_SEED
)
results.append({
    "template_type": "Value",
    "target": "Operand 0",
    "accuracy_val": res_val_a["acc_val"],
    "accuracy_test": res_val_a["acc_test"],
    "n_train": res_val_a["n_train"],
    "n_val": res_val_a["n_val"],
    "n_test": res_val_a["n_test"],
    "n_classes": res_val_a["n_classes"],
})

print("\n[Value template] Operand 1 (b)...")
res_val_b = train_val_test_on_same_data(
    traces, train_b_u8,
    train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO, seed=SPLIT_SEED
)
results.append({
    "template_type": "Value",
    "target": "Operand 1",
    "accuracy_val": res_val_b["acc_val"],
    "accuracy_test": res_val_b["acc_test"],
    "n_train": res_val_b["n_train"],
    "n_val": res_val_b["n_val"],
    "n_test": res_val_b["n_test"],
    "n_classes": res_val_b["n_classes"],
})

print("\n[Value template] Output (v)...")
res_val_v = train_val_test_on_same_data(
    traces, train_v_u8,
    train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO, seed=SPLIT_SEED
)
results.append({
    "template_type": "Value",
    "target": "Output",
    "accuracy_val": res_val_v["acc_val"],
    "accuracy_test": res_val_v["acc_test"],
    "n_train": res_val_v["n_train"],
    "n_val": res_val_v["n_val"],
    "n_test": res_val_v["n_test"],
    "n_classes": res_val_v["n_classes"],
})

# -------------------------
# Hamming weight templates
# -------------------------
print("\n[Hamming weight template] Operand 0 (a)...")
res_hw_a = train_val_test_on_same_data(
    traces, train_a_hw,
    train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO, seed=SPLIT_SEED
)
results.append({
    "template_type": "Hamming weight",
    "target": "Operand 0",
    "accuracy_val": res_hw_a["acc_val"],
    "accuracy_test": res_hw_a["acc_test"],
    "n_train": res_hw_a["n_train"],
    "n_val": res_hw_a["n_val"],
    "n_test": res_hw_a["n_test"],
    "n_classes": res_hw_a["n_classes"],
})

print("\n[Hamming weight template] Operand 1 (b)...")
res_hw_b = train_val_test_on_same_data(
    traces, train_b_hw,
    train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO, seed=SPLIT_SEED
)
results.append({
    "template_type": "Hamming weight",
    "target": "Operand 1",
    "accuracy_val": res_hw_b["acc_val"],
    "accuracy_test": res_hw_b["acc_test"],
    "n_train": res_hw_b["n_train"],
    "n_val": res_hw_b["n_val"],
    "n_test": res_hw_b["n_test"],
    "n_classes": res_hw_b["n_classes"],
})

print("\n[Hamming weight template] Output (v)...")
res_hw_v = train_val_test_on_same_data(
    traces, train_v_hw,
    train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO, seed=SPLIT_SEED
)
results.append({
    "template_type": "Hamming weight",
    "target": "Output",
    "accuracy_val": res_hw_v["acc_val"],
    "accuracy_test": res_hw_v["acc_test"],
    "n_train": res_hw_v["n_train"],
    "n_val": res_hw_v["n_val"],
    "n_test": res_hw_v["n_test"],
    "n_classes": res_hw_v["n_classes"],
})

results_df = pd.DataFrame(results)
print("\n" + "="*80)
print("ALL RESULTS (Validation & Test)")
print("="*80)
display(results_df)

## 8. Table 2-style summary

In [ ]:
# Validation 결과
print("\n" + "="*80)
print("VALIDATION ACCURACY (by template type and target)")
print("="*80)
table_val = (
    results_df
    .pivot(index="target", columns="template_type", values="accuracy_val")
    .rename(columns={
        "Value": "Value template",
        "Hamming weight": "Hamming weight template"
    })
    .loc[["Operand 0", "Operand 1", "Output"]]
)
display(table_val.style.format("{:.4f}"))

# Test 결과
print("\n" + "="*80)
print("TEST ACCURACY (by template type and target)")
print("="*80)
table_test = (
    results_df
    .pivot(index="target", columns="template_type", values="accuracy_test")
    .rename(columns={
        "Value": "Value template",
        "Hamming weight": "Hamming weight template"
    })
    .loc[["Operand 0", "Operand 1", "Output"]]
)
display(table_test.style.format("{:.4f}"))

# 전체 요약
print("\n" + "="*80)
print("SUMMARY: Val vs Test Accuracy")
print("="*80)
summary = results_df.groupby("template_type")[["accuracy_val", "accuracy_test"]].mean()
print(summary.to_string())